# Genie Agent — Sample Demand Queries

This notebook demonstrates the **Genie Conversation API** to ask business
questions about demand forecasts programmatically. Each cell submits a
natural-language question, waits for SQL generation and execution, and
displays the result.

**Prerequisites:** Run `setup_genie_agent` first to create the Genie space.

In [0]:
import time
from databricks.sdk import WorkspaceClient

GENIE_SPACE_ID = "01f1c22c3f621207842ec9d6577c22c7"
w = WorkspaceClient()

def ask_genie(question: str, space_id: str = GENIE_SPACE_ID) -> dict:
    """Submit a question to the Genie Agent and wait for the answer."""
    conv = w.genie.start_conversation(space_id=space_id, content=question)
    for _ in range(60):
        msg = w.genie.get_message(
            space_id=space_id,
            conversation_id=conv.conversation_id,
            message_id=conv.message_id,
        )
        if msg.status in ("COMPLETED", "FAILED"):
            break
        time.sleep(2)
    result = {"status": msg.status, "question": question}
    if hasattr(msg, 'attachments') and msg.attachments:
        for att in msg.attachments:
            if hasattr(att, 'query'):
                result["sql"] = att.query.query
                if hasattr(att.query, 'result'):
                    result["columns"] = att.query.result.columns
                    result["data"] = att.query.result.data_array
            if hasattr(att, 'text'):
                result["narrative"] = att.text.content
    return result

def display_genie(result: dict):
    print(f"Q: {result['question']}")
    print(f"Status: {result['status']}")
    if 'sql' in result:
        print(f"SQL: {result['sql'][:200]}...")
    if 'narrative' in result:
        print(f"Answer: {result['narrative']}")
    if 'data' in result and result['data']:
        import pandas as pd
        col_names = [c.name for c in result['columns']] if result.get('columns') else None
        df = pd.DataFrame(result['data'], columns=col_names)
        display(df.head(10))

In [0]:
r = ask_genie("Which model has the lowest average sMAPE across all time series?")
display_genie(r)

Q: Which model has the lowest average sMAPE across all time series?
Status: MessageStatus.COMPLETED

Generated SQL:
WITH model_avg AS (
  SELECT
    `model`,
    AVG(`metric_value`) AS `avg_smape`
  FROM `mmf_demo_aym`.`m4_advanced`.`monthly_evaluation_output`
  WHERE `metric_name` ILIKE '%smape%'
    AND `model` IS NOT NULL
    AND `metric_value` IS NOT NULL
  GROUP BY `model`
), ranked AS (
  SELECT
    `model`,
    `avg_smape`,
    RANK() OVER (ORDER BY `avg_smape` ASC) AS `rnk`
  FROM model_avg
)
SELECT
  `model`,
  `avg_smape`
FROM ranked
WHERE `rnk` <= 1
ORDER BY `avg_smape` ASC, `model` ASC

Answer:
The model with the **lowest average sMAPE** across all time series is **StatsForecastAutoCES**, with an average sMAPE of **0.061535488464333266**. Based on the ranked result, **StatsForecastAutoCES** is the top-performing model on this accuracy measure in the available data.


In [0]:
r = ask_genie("Compare the top 5 models for time series M1 by average metric value")
display_genie(r)

Q: Compare the top 5 models for time series M1 by average metric value
Status: MessageStatus.COMPLETED

Generated SQL:
WITH model_metric_avg AS ( SELECT `metric_name`, `model`, AVG(`metric_value`) AS `avg_metric_value` FROM `mmf_demo_aym`.`m4_advanced`.`monthly_evaluation_output` WHERE `unique_id` LIKE 'M1%' AND `model` IS NOT NULL AND `metric_name` IS NOT NULL AND `metric_value` IS NOT NULL GROUP BY `metric_name`, `model` ), ranked_models AS ( SELECT `metric_name`, `model`, `avg_metric_value`, RANK() OVER (PARTITION BY `metric_name` ORDER BY `avg_metric_value` DESC) AS `model_rank` FROM model_metric_avg ) SELECT `metric_name`, `model`, `avg_metric_value`, `model_rank` FROM ranked_models WHERE `model_rank` <= 5 ORDER BY `metric_name`, `model_rank`, `model`

Answer:
For time series **M1**, the top 5 models by average metric value for **smape** are ranked from highest to lowest as:
- **StatsForecastCrostonSBA** — **0.0994**
- **StatsForecastCrostonClassic** — **0.0771**
- **StatsForecast

In [0]:
r = ask_genie("How many unique time series are in the training data, and what is the date range?")
display_genie(r)

Q: How many unique time series are in the training data and what is the date range?
Status: MessageStatus.COMPLETED

Generated SQL:
SELECT COUNT(DISTINCT `unique_id`) AS `unique_time_series`, MIN(`ds`) AS `start_date`, MAX(`ds`) AS `end_date`
FROM `mmf_demo_aym`.`m4_advanced`.`m4_monthly_train`
WHERE `unique_id` IS NOT NULL AND `ds` IS NOT NULL

Answer:
The training data contains **99** unique time series, with dates ranging from **2018-01-31** to **2022-12-31**. Based on the full result shown, the training period spans just under **5 years**.


In [0]:
r = ask_genie("What percentage of models have non-empty prediction intervals in the scoring output?")
display_genie(r)

Q: What percentage of models have non-empty prediction intervals in the scoring output?
Status: MessageStatus.COMPLETED

Generated SQL:
SELECT try_divide(100.0 * SUM(CASE WHEN `forecast_lower` IS NOT NULL AND `forecast_upper` IS NOT NULL AND size(`forecast_lower`) > 0 AND size(`forecast_upper`) > 0 THEN 1 ELSE 0 END),COUNT(*)) AS `percentage_with_non_empty_prediction_intervals`
FROM `mmf_demo_aym`.`m4_advanced`.`monthly_scoring_output`

Answer:
**0.00%** of models in `monthly_scoring_output` have non-empty prediction intervals in the scoring output, based on `percentage_with_non_empty_prediction_intervals = 0.00000000000000`. This means **no models** in the result set have both `forecast_lower` and `forecast_upper` populated with non-empty values.
